In [5]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd

df = pd.read_csv("../data/raw/globalmech_data.csv",
                 parse_dates=["Date"],
                 index_col="Date")

In [10]:
targets = ["Dom_Cap", "Dom_Spares", "Exp_Cap", "Exp_Spares"]

df_fe = df.copy()

for col in targets:
    df_fe[f"{col}_lag1"] = df_fe[col].shift(1)
    df_fe[f"{col}_lag3"] = df_fe[col].shift(3)

for col in ["Dom_Cap", "Dom_Spares"]:
    df_fe[f"{col}_roll3"] = df_fe[col].rolling(3).mean()

df_fe["month"] = df_fe.index.month
df_fe["quarter"] = df_fe.index.quarter

df_fe.dropna(inplace=True)

Model 1 — SARIMAX (CORE MODEL)

In [11]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

target = "Exp_Spares"

y = df_fe[target]
X = df_fe[["USD_INR", "Install_Base", "Shipping_Index"]]

model = SARIMAX(y, exog=X, order=(1,1,1), seasonal_order=(1,1,1,12))
results = model.fit()

print(results.summary())

/Users/ash/Desktop/globalmech_forecasting/datascience/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/ash/Desktop/globalmech_forecasting/datascience/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/ash/Desktop/globalmech_forecasting/datascience/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                                     SARIMAX Results                                      
Dep. Variable:                         Exp_Spares   No. Observations:                   69
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 12)   Log Likelihood                -248.504
Date:                            Wed, 08 Apr 2026   AIC                            513.008
Time:                                    04:00:09   BIC                            529.211
Sample:                                04-01-2019   HQIC                           519.290
                                     - 12-01-2024                                         
Covariance Type:                              opg                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
USD_INR           -2.4470      3.398     -0.720      0.471      -9.107       4.213
Install_Base      -0.22

Model 2 — Machine Learning (XGBoost)

<h3>Evaluation :

<h3> BUSINESS INSIGHTS: </h3>

* Export spares ↑ when USD ↑
* Domestic spares strongly tied to install base
* Shipping delays negatively impact export capital